# TailCluster and Exceptional Transition Subgroups

Runs an independent longitudinal subgroup discovery pipeline on World Bank life expectancy and GDP per capita. Outputs include row labels, spans, RF rules, and clean tree exports.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "transition_subgroup_discovery").exists():
    WORKSPACE_ROOT = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "transition_subgroup_discovery":
    WORKSPACE_ROOT = cwd.parents[1]
else:
    raise RuntimeError("Launch from the workspace root or transition_subgroup_discovery/notebooks.")

REPO_ROOT = WORKSPACE_ROOT / "transition_subgroup_discovery"
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

CONFIG_PATH = REPO_ROOT / "configs" / "world_bank.json"
OUTPUT_DIR = REPO_ROOT / "outputs" / "world_bank"
print(WORKSPACE_ROOT)

In [ ]:
from tsd import run_pipeline

run_pipeline(CONFIG_PATH, WORKSPACE_ROOT)

In [ ]:
import pandas as pd

summary = pd.read_csv(OUTPUT_DIR / "run_summary.csv")
summary

In [ ]:
for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition"]:
        print("\n" + "=" * 100)
        print(f"{target} / {method} / RF metrics")
        print("=" * 100)
        metrics = pd.read_json(OUTPUT_DIR / target / method / "rf" / "rule_metrics.json", typ="series")
        display(metrics.to_frame("value"))
        print(f"{target} / {method} / positive rules")
        display(pd.read_csv(OUTPUT_DIR / target / method / "rf" / "rules.csv"))

In [ ]:
for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["trajtrack", "trajtrack_binary"]:
        traj_dir = OUTPUT_DIR / target / method
        print("\n" + "=" * 100)
        print(f"{target} / {method} / divergence-ranked rules")
        print("=" * 100)
        metrics_path = traj_dir / "metrics.json"
        rules_path = traj_dir / "rules.csv"
        text_path = traj_dir / "rules_natural_language.txt"
        if metrics_path.exists():
            display(pd.read_json(metrics_path, typ="series").to_frame("value"))
        if rules_path.exists():
            display(pd.read_csv(rules_path))
        if text_path.exists():
            print(text_path.read_text(encoding="utf-8"))


In [ ]:
for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition"]:
        tree_path = OUTPUT_DIR / target / method / "rf" / "trees" / "tree_00_idx_000.txt"
        if not tree_path.exists():
            tree_path = sorted((OUTPUT_DIR / target / method / "rf" / "trees").glob("tree_00*.txt"))[0]
        print("\n" + "=" * 100)
        print(f"{target} / {method} / clean tree: {tree_path.name}")
        print("=" * 100)
        print(tree_path.read_text(encoding="utf-8"))

In [ ]:
for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition"]:
        counts_path = OUTPUT_DIR / target / method / "transition_counts.csv"
        if counts_path.exists():
            print("\n" + "=" * 100)
            print(f"{target} / {method} / transition counts")
            print("=" * 100)
            display(pd.read_csv(counts_path))


In [ ]:
for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition"]:
        summary_path = OUTPUT_DIR / target / method / "transition_summary.csv"
        if summary_path.exists():
            print("\n" + "=" * 100)
            print(f"{target} / {method} / numeric transition summary")
            print("=" * 100)
            display(pd.read_csv(summary_path))


In [ ]:
from IPython.display import Image, display

for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition"]:
        print(f"{target} / {method} / KDE")
        display(Image(filename=str(OUTPUT_DIR / target / method / "interestingness_kde.png")))

In [ ]:
from IPython.display import Image, display

for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["trajtrack", "trajtrack_binary"]:
        plot_dir = OUTPUT_DIR / target / method / "plots"
        print("\n" + "=" * 100)
        print(f"{target} / {method} / trajectory and KDE diagnostics")
        print("=" * 100)
        for plot_path in sorted(plot_dir.glob("rule_*.png")):
            print(plot_path.name)
            display(Image(filename=str(plot_path)))


In [ ]:
from IPython.display import Image, display

def display_forest_trees(tree_dir, heading):
    manifest_path = tree_dir / "manifest.csv"
    print("\n" + "=" * 100)
    print(heading)
    print("=" * 100)
    if not manifest_path.exists():
        print(f"No tree manifest found: {manifest_path}")
        return
    manifest = pd.read_csv(manifest_path)
    for row in manifest.itertuples(index=False):
        print(f"Tree {row.tree_rank} | forest index {row.forest_index} | depth {row.depth} | leaves {row.leaves}")
        display(Image(filename=str(tree_dir / row.png)))

for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition"]:
        display_forest_trees(
            OUTPUT_DIR / target / method / "rf" / "forest_trees",
            f"{target} / {method} / selected compact forest",
        )
    display_forest_trees(
        OUTPUT_DIR / target / "trajtrack" / "forest_trees",
        f"{target} / trajtrack / selected surrogate forest",
    )
    display_forest_trees(
        OUTPUT_DIR / target / "trajtrack_binary" / "forest_trees",
        f"{target} / trajtrack_binary / selected classifier forest",
    )


In [ ]:
# Optional: transition-delta diagnostics for existing Interpretable_SD TimeTribes outputs.
# This does not run TimeTribes; it only visualizes row_labels.csv if already present.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

TIMETRIBES_CANDIDATES = {
    "life_expectancy": {
        "path": WORKSPACE_ROOT / "Interpretable_SD" / "outputs_old" / "world_bank_full_paper_grade" / "life_expectancy_spectrum" / "timetribes_windowed" / "row_labels.csv",
        "target_col": "life_expectancy_at_birth",
        "two_tailed": True,
    },
    "gdp_per_capita": {
        "path": WORKSPACE_ROOT / "Interpretable_SD" / "outputs_old" / "world_bank_full_paper_grade" / "gdp_per_capita_variable_windows" / "timetribes_windowed" / "row_labels.csv",
        "target_col": "gdp_per_capita",
        "two_tailed": True,
    },
}

def _label_col(df):
    for col in ["is_interesting", "is_interesting_subgroup", "label", "target"]:
        if col in df.columns:
            return col
    bool_cols = [c for c in df.columns if "interesting" in c.lower()]
    return bool_cols[0] if bool_cols else None

def _add_transition_fields(df, id_col, date_col, target_col, two_tailed=True):
    out = df.copy()
    if "_year" not in out.columns:
        out["_year"] = pd.to_datetime(out[date_col], errors="coerce").dt.year
    out = out.sort_values([id_col, "_year"]).copy()
    y = pd.to_numeric(out[target_col], errors="coerce")
    out["next_target"] = out.groupby(id_col, sort=False)[target_col].shift(-1)
    out["transition_delta"] = pd.to_numeric(out["next_target"], errors="coerce") - y
    delta = pd.to_numeric(out["transition_delta"], errors="coerce")
    center = delta.groupby(out["_year"]).transform("median")
    mad = (delta - center).abs().groupby(out["_year"]).transform("median")
    global_mad = float((delta - delta.median()).abs().median() or delta.std() or 1.0)
    out["transition_z"] = (delta - center) / (1.4826 * mad.replace(0, np.nan).fillna(global_mad))
    out["transition_score"] = out["transition_z"].abs() if two_tailed else out["transition_z"]
    return out

def _plot_timetribes_transition_kde(df, label_col, value_col, title):
    all_values = pd.to_numeric(df[value_col], errors="coerce").dropna()
    interesting_values = pd.to_numeric(df.loc[df[label_col].astype(bool), value_col], errors="coerce").dropna()
    non_values = pd.to_numeric(df.loc[~df[label_col].astype(bool), value_col], errors="coerce").dropna()
    plt.figure(figsize=(12, 7))
    if len(all_values) > 1:
        sns.kdeplot(all_values, color="blue", fill=True, alpha=0.25, label="Original", bw_adjust=1.1, common_norm=False)
    if len(non_values) > 1:
        sns.kdeplot(non_values, color="green", fill=True, alpha=0.25, label="Non-Interesting", bw_adjust=1.1, common_norm=False)
    if len(interesting_values) > 1:
        sns.kdeplot(interesting_values, color="red", fill=True, alpha=0.25, label="Interesting", bw_adjust=1.1, common_norm=False)
    plt.title(title)
    plt.xlabel(value_col)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

for target, spec in TIMETRIBES_CANDIDATES.items():
    path = spec["path"]
    if not path.exists():
        print(f"Skipping TimeTribes {target}: {path} not found")
        continue
    df = pd.read_csv(path)
    label_col = _label_col(df)
    if label_col is None or spec["target_col"] not in df.columns:
        print(f"Skipping TimeTribes {target}: missing label or target column")
        continue
    df = _add_transition_fields(df, "country", "date", spec["target_col"], spec["two_tailed"])
    print("\n" + "=" * 100)
    print(f"TimeTribes / {target} / transition diagnostics")
    print("=" * 100)
    display(df.groupby(label_col)[["transition_delta", "transition_z", "transition_score"]].describe())
    for value_col in ["transition_delta", "transition_z", "transition_score"]:
        _plot_timetribes_transition_kde(df, label_col, value_col, f"TimeTribes {target}: {value_col}")


In [ ]:
from IPython.display import Image, display

for target in ["life_expectancy", "gdp_per_capita"]:
    plot_dir = OUTPUT_DIR / target / "numeric_transition" / "rf" / "rule_plots"
    print("\n" + "=" * 100)
    print(f"{target} / numeric_transition / rule-level KDE diagnostics")
    print("=" * 100)
    for plot_path in sorted(plot_dir.glob("*.png")):
        print(plot_path.name)
        display(Image(filename=str(plot_path)))


In [ ]:
from IPython.display import Image, display

for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["tail_cluster", "numeric_transition", "trajtrack", "trajtrack_binary"]:
        method_dir = OUTPUT_DIR / target / method
        overview_dir = method_dir if method in {"trajtrack", "trajtrack_binary"} else method_dir / "rf"
        image_path = overview_dir / "subgroup_distribution.png"
        overlay_path = overview_dir / "subgroup_distribution_overlay.png"
        axis_path = overview_dir / "subgroup_distribution_axis_selection.csv"
        rules_path = overview_dir / "subgroup_distribution_rules.txt"
        evidence_path = overview_dir / "subgroup_evidence_summary.png"
        evidence_key_path = overview_dir / "subgroup_evidence_summary.txt"
        print("\n" + "=" * 100)
        print(f"{target} / {method} / individual subgroup distributions")
        print("=" * 100)
        if image_path.exists():
            display(Image(filename=str(image_path)))
        print(f"{target} / {method} / overlaid subgroup distributions")
        if overlay_path.exists():
            display(Image(filename=str(overlay_path)))
        if axis_path.exists():
            display(pd.read_csv(axis_path))
        if rules_path.exists():
            print(rules_path.read_text(encoding="utf-8"))
        if method in {"trajtrack", "trajtrack_binary"} and evidence_path.exists():
            print(f"{target} / {method} / compact subgroup evidence summary")
            display(Image(filename=str(evidence_path)))
        if method in {"trajtrack", "trajtrack_binary"} and evidence_key_path.exists():
            print(evidence_key_path.read_text(encoding="utf-8"))


In [ ]:
from IPython.display import Image, display

for target in ["life_expectancy", "gdp_per_capita"]:
    for method in ["trajtrack", "trajtrack_binary"]:
        plot_path = OUTPUT_DIR / target / method / "performance_vs_questions.png"
        print(f"{target} / {method} forest performance vs questions")
        if plot_path.exists():
            display(Image(filename=str(plot_path)))
